<a href="https://colab.research.google.com/github/RonazAbbasi/ml-zoomcamp2026/blob/main/Homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

# 1. Load the correct 2026 dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

print("Dataset rows and columns:", df.shape)
print("Dataset columns:", df.columns.tolist())

# 2. Fill missing values according to the homework
target = "converted"

categorical = df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical = df.select_dtypes(include=["number"]).columns.tolist()
numerical = [col for col in numerical if col != target]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

# 3. Split the data using the exact random_state from the homework
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train[target].values
y_val = df_val[target].values

X_train_df = df_train.drop(columns=[target])
X_val_df = df_val.drop(columns=[target])

print("\nQ1: ROC AUC feature importance")

features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

auc_scores = {}

for col in features:
    auc = roc_auc_score(y_train, X_train_df[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -X_train_df[col])
    auc_scores[col] = auc
    print(col, round(auc, 3))

print("Q1 answer:", max(auc_scores, key=auc_scores.get))

# 4. Train Logistic Regression
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(X_train_df.to_dict(orient="records"))
X_val = dv.transform(X_val_df.to_dict(orient="records"))

model = LogisticRegression(
    solver="liblinear", C=1.0, max_iter=1000
)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]

print("\nQ2: Validation ROC AUC")
auc = roc_auc_score(y_val, y_pred)
print("AUC:", round(auc, 3))

# 5. Precision, Recall and F1 for thresholds 0.00 to 1.00
thresholds = np.arange(0.0, 1.001, 0.01)

precisions = []
recalls = []
f1_scores = []

for threshold in thresholds:
    pred = (y_pred >= threshold).astype(int)

    p = precision_score(y_val, pred, zero_division=0)
    r = recall_score(y_val, pred, zero_division=0)
    f = f1_score(y_val, pred, zero_division=0)

    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f)

# Q3: Threshold where precision and recall are closest
valid_indices = [
    i for i in range(len(thresholds))
    if not (precisions[i] == 0 and recalls[i] == 0)
]

best_intersection = min(
    valid_indices,
    key=lambda i: abs(precisions[i] - recalls[i])
)

print("\nQ3: Precision and Recall")
print("Threshold:", round(thresholds[best_intersection], 2))
print("Precision:", round(precisions[best_intersection], 3))
print("Recall:", round(recalls[best_intersection], 3))

# Q4: Threshold with maximum F1
best_f1_index = int(np.argmax(f1_scores))

print("\nQ4: F1 score")
print("Best threshold:", round(thresholds[best_f1_index], 2))
print("Maximum F1:", round(f1_scores[best_f1_index], 3))

# 6. Five-fold cross-validation on df_full_train
df_full_train = df_full_train.reset_index(drop=True)
y_full = df_full_train[target].values
X_full_df = df_full_train.drop(columns=[target])

kf = KFold(n_splits=5, shuffle=True, random_state=1)

print("\nQ5: 5-Fold Cross-validation")

scores = []

for train_idx, val_idx in kf.split(X_full_df):
    X_fold_train_df = X_full_df.iloc[train_idx]
    X_fold_val_df = X_full_df.iloc[val_idx]

    y_fold_train = y_full[train_idx]
    y_fold_val = y_full[val_idx]

    fold_dv = DictVectorizer(sparse=True)

    X_fold_train = fold_dv.fit_transform(
        X_fold_train_df.to_dict(orient="records")
    )
    X_fold_val = fold_dv.transform(
        X_fold_val_df.to_dict(orient="records")
    )

    fold_model = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000
    )
    fold_model.fit(X_fold_train, y_fold_train)

    fold_pred = fold_model.predict_proba(X_fold_val)[:, 1]
    scores.append(roc_auc_score(y_fold_val, fold_pred))

print("Fold AUC scores:", [round(s, 3) for s in scores])
print("Mean AUC:", round(np.mean(scores), 3))
print("Standard deviation:", round(np.std(scores), 3))

# 7. Hyperparameter tuning
print("\nQ6: Hyperparameter Tuning")

results = []

for C in [0.000001, 0.001, 1]:
    c_scores = []

    for train_idx, val_idx in kf.split(X_full_df):
        X_fold_train_df = X_full_df.iloc[train_idx]
        X_fold_val_df = X_full_df.iloc[val_idx]

        y_fold_train = y_full[train_idx]
        y_fold_val = y_full[val_idx]

        fold_dv = DictVectorizer(sparse=True)

        X_fold_train = fold_dv.fit_transform(
            X_fold_train_df.to_dict(orient="records")
        )
        X_fold_val = fold_dv.transform(
            X_fold_val_df.to_dict(orient="records")
        )

        fold_model = LogisticRegression(
            solver="liblinear", C=C, max_iter=1000
        )
        fold_model.fit(X_fold_train, y_fold_train)

        fold_pred = fold_model.predict_proba(X_fold_val)[:, 1]
        c_scores.append(roc_auc_score(y_fold_val, fold_pred))

    mean_score = np.mean(c_scores)
    std_score = np.std(c_scores)

    results.append((C, mean_score, std_score))

    print(
        "C =", C,
        "| Mean AUC =", round(mean_score, 3),
        "| Std =", round(std_score, 3)
    )

best_result = sorted(
    results,
    key=lambda item: (-round(item[1], 3), round(item[2], 3), item[0])
)[0]

print("Q6 answer: Best C =", best_result[0])

Dataset rows and columns: (5000, 9)
Dataset columns: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

Q1: ROC AUC feature importance
lead_score 0.788
number_of_courses_viewed 0.723
interaction_count 0.765
annual_income 0.608
Q1 answer: lead_score

Q2: Validation ROC AUC
AUC: 0.732

Q3: Precision and Recall
Threshold: 0.63
Precision: 0.733
Recall: 0.725

Q4: F1 score
Best threshold: 0.41
Maximum F1: 0.758

Q5: 5-Fold Cross-validation
Fold AUC scores: [np.float64(0.731), np.float64(0.73), np.float64(0.736), np.float64(0.741), np.float64(0.721)]
Mean AUC: 0.732
Standard deviation: 0.007

Q6: Hyperparameter Tuning
C = 1e-06 | Mean AUC = 0.617 | Std = 0.019
C = 0.001 | Mean AUC = 0.749 | Std = 0.01
C = 1 | Mean AUC = 0.732 | Std = 0.007
Q6 answer: Best C = 0.001
